# Brazilian Credit Derivatives — End-to-End Calibration

**From market quotes to CLN and TRS pricing in 6 steps:**

1. Build CDI discount curve from DI futures (realistic ~14.75% Selic, inverted curve)
2. Price NTN-F bonds → extract yields
3. Bootstrap piecewise hazard rates from corporate bond prices
4. Build CDS curve from calibrated survival curve
5. Price a CLN on a Brazilian corporate
6. Price a TRS on the bond

All using BUS/252, São Paulo calendar, CDI compounding.

In [ ]:
import sys; sys.path.insert(0, "..")
import math, numpy as np, matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta
from dataclasses import dataclass

from pricebook.viz import configure_theme, greeks_profile, sensitivity_grid
from pricebook.core.day_count import DayCountConvention, year_fraction
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.survival_curve import SurvivalCurve
from pricebook.core.schedule import Frequency
from pricebook.fixed_income.brazilian import (
    build_cdi_curve_from_di, DIFuture, DISwap, LFTBond,
    _bus_days, _di_discount_factor, _di_rate_from_df,
)
from pricebook.fixed_income.sovereign_bonds import create_sovereign_bond, create_sovereign_zero
from pricebook.credit.bond_hazard_bootstrap import bootstrap_hazard_from_bonds, BondInput
from pricebook.credit.cds import CDS
from pricebook.credit.cln import CreditLinkedNote
from pricebook.equity.trs import TotalReturnSwap

configure_theme(dark=False)
REF = date(2024, 11, 4)  # Monday
print("Setup complete — REF:", REF)

## Step 1: CDI Discount Curve — Realistic Brazilian Term Structure

Current Selic at 14.75%. The curve is **inverted** (front end above long end) reflecting market expectations of future rate cuts.

$$DF(T) = \frac{1}{(1 + DI)^{bd/252}}$$

In [ ]:
# Realistic DI futures strip — Nov 2024 market snapshot
# Selic: 14.75%, inverted curve expecting cuts to ~11% by 2027
di_strip = [
    {"maturity": date(2025, 1, 2),  "rate": 0.1475, "bus_days": _bus_days(REF, date(2025, 1, 2))},
    {"maturity": date(2025, 4, 1),  "rate": 0.1460, "bus_days": _bus_days(REF, date(2025, 4, 1))},
    {"maturity": date(2025, 7, 1),  "rate": 0.1430, "bus_days": _bus_days(REF, date(2025, 7, 1))},
    {"maturity": date(2025, 10, 1), "rate": 0.1390, "bus_days": _bus_days(REF, date(2025, 10, 1))},
    {"maturity": date(2026, 1, 2),  "rate": 0.1340, "bus_days": _bus_days(REF, date(2026, 1, 2))},
    {"maturity": date(2026, 7, 1),  "rate": 0.1280, "bus_days": _bus_days(REF, date(2026, 7, 1))},
    {"maturity": date(2027, 1, 4),  "rate": 0.1220, "bus_days": _bus_days(REF, date(2027, 1, 4))},
    {"maturity": date(2027, 7, 1),  "rate": 0.1180, "bus_days": _bus_days(REF, date(2027, 7, 1))},
    {"maturity": date(2028, 1, 3),  "rate": 0.1150, "bus_days": _bus_days(REF, date(2028, 1, 3))},
    {"maturity": date(2029, 1, 2),  "rate": 0.1120, "bus_days": _bus_days(REF, date(2029, 1, 2))},
    {"maturity": date(2030, 1, 2),  "rate": 0.1100, "bus_days": _bus_days(REF, date(2030, 1, 2))},
    {"maturity": date(2031, 1, 2),  "rate": 0.1090, "bus_days": _bus_days(REF, date(2031, 1, 2))},
    {"maturity": date(2034, 1, 2),  "rate": 0.1080, "bus_days": _bus_days(REF, date(2034, 1, 2))},
]

cdi_curve = build_cdi_curve_from_di(REF, di_strip)

print(f"{'Maturity':<14s} {'DI Rate':<10s} {'Bus Days':<10s} {'DF':<12s}")
print("-" * 46)
for c in di_strip:
    df = cdi_curve.df(c["maturity"])
    print(f"{str(c['maturity']):<14s} {c['rate']*100:>7.2f}%   {c['bus_days']:>5d}     {df:>10.6f}")

In [ ]:
# Plot CDI term structure — inverted curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

years = [c["bus_days"] / 252 for c in di_strip]
rates = [c["rate"] * 100 for c in di_strip]
dfs = [cdi_curve.df(c["maturity"]) for c in di_strip]

ax1.plot(years, rates, 'b-o', linewidth=2, markersize=6)
ax1.set_xlabel('Maturity (years)', fontsize=12)
ax1.set_ylabel('DI Rate (%)', fontsize=12)
ax1.set_title('CDI Term Structure — Inverted (Nov 2024)', fontsize=13)
ax1.axhline(y=14.75, color='red', linestyle='--', alpha=0.5, label='Selic target (14.75%)')
ax1.annotate('Rate cuts expected →', xy=(3, 12.5), fontsize=10, color='green',
             arrowprops=dict(arrowstyle='->', color='green'), xytext=(1, 13.5))
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

ax2.plot(years, dfs, 'g-o', linewidth=2, markersize=6)
ax2.set_xlabel('Maturity (years)', fontsize=12)
ax2.set_ylabel('Discount Factor', fontsize=12)
ax2.set_title('CDI Discount Factors', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('brazilian_cdi_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Curve shape: inverted ({rates[0]:.1f}% → {rates[-1]:.1f}%), expecting {rates[0]-rates[-1]:.0f}bp of cuts")

## Step 2: NTN-F Bond Pricing

NTN-F: 10% semi-annual coupon, BUS/252, face R\$ 1,000.

At current CDI levels (~11-14%), a 10% coupon bond trades **below par**.

In [ ]:
# Price NTN-F bonds at standard maturities
bond_mats = {
    "NTN-F 2027": date(2027, 1, 1),
    "NTN-F 2029": date(2029, 1, 1),
    "NTN-F 2031": date(2031, 1, 1),
    "NTN-F 2035": date(2035, 1, 1),
}

ntn_f_prices = {}
print(f"{'Bond':<16s} {'Dirty Price':<14s} {'Yield':<10s} {'Duration*':<10s}")
print("-" * 50)
for name, mat in bond_mats.items():
    bond = create_sovereign_bond("NTN_F", REF, mat, 0.10)
    price = bond.dirty_price(cdi_curve)
    ntn_f_prices[name] = price
    bd = _bus_days(REF, mat)
    y = _di_rate_from_df(price / 100, bd) if price > 0 else 0
    dur = bd / 252 / (1 + y) if y > -1 else 0  # modified duration approx
    print(f"  {name:<14s} {price:>10.4f}     {y*100:>6.2f}%   {dur:>6.2f}Y")

## Step 3: Piecewise Hazard Rate Bootstrap from Corporate Bond Prices

We observe corporate bond prices at 3Y, 5Y, 7Y, 10Y maturities. Each trades at a **credit spread** over the sovereign (NTN-F) curve.

The piecewise-constant hazard rate model:

$$Q(T) = \exp\left(-\sum_{i} h_i \cdot \Delta t_i\right)$$

where $h_i$ is constant between pillars. We bootstrap $h_1, h_2, h_3, h_4$ sequentially to reprice each bond exactly.

In [ ]:
# Corporate bond prices — spread over sovereign
# Brazilian BBB corporate: ~200-400bp credit spread
corporate_bonds = [
    BondInput(maturity=date(2027, 7, 1), coupon=0.13, market_price=98.5, recovery=0.25,
              frequency=2, liquidity_spread_bp=20),
    BondInput(maturity=date(2029, 7, 1), coupon=0.13, market_price=95.0, recovery=0.25,
              frequency=2, liquidity_spread_bp=15),
    BondInput(maturity=date(2031, 7, 1), coupon=0.125, market_price=91.0, recovery=0.25,
              frequency=2, liquidity_spread_bp=10),
    BondInput(maturity=date(2034, 7, 1), coupon=0.12, market_price=85.0, recovery=0.25,
              frequency=2, liquidity_spread_bp=10),
]

# Bootstrap piecewise hazard rates
hazard_result = bootstrap_hazard_from_bonds(REF, corporate_bonds, cdi_curve)

print("Piecewise Hazard Rate Bootstrap:")
print(f"{'Pillar':<14s} {'Hazard (%)':<12s} {'Survival':<12s} {'CDS Spread*':<14s}")
print("-" * 52)
for d, h in zip(hazard_result.pillar_dates, hazard_result.pillar_hazards):
    q = hazard_result.survival_curve.survival(d)
    cds_approx = (1 - 0.25) * h * 10000  # (1-R) × h in bp
    print(f"  {str(d):<12s} {h*100:>8.3f}     {q*100:>8.2f}%   {cds_approx:>8.0f} bp")

print(f"\nCalibration quality: RMSE = {hazard_result.rmse_bp:.2f} bp, max error = {hazard_result.max_error_bp:.2f} bp")
print(f"Fitted vs market prices:")
for i, bond in enumerate(corporate_bonds):
    print(f"  {bond.maturity}: market={bond.market_price:.2f}, model={hazard_result.fitted_prices[i]:.2f}, "
          f"error={hazard_result.residuals_bp[i]:.1f} bp")

In [ ]:
# Plot piecewise hazard rates + survival curve
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Piecewise hazard rates (step function)
ax = axes[0]
pillar_years = [_bus_days(REF, d) / 252 for d in hazard_result.pillar_dates]
hazards_bp = [h * 10000 for h in hazard_result.pillar_hazards]
# Step plot
for i in range(len(pillar_years)):
    x_start = 0 if i == 0 else pillar_years[i-1]
    x_end = pillar_years[i]
    ax.plot([x_start, x_end], [hazards_bp[i], hazards_bp[i]], 'r-', linewidth=2.5)
    if i > 0:
        ax.plot([x_start, x_start], [hazards_bp[i-1], hazards_bp[i]], 'r--', linewidth=1, alpha=0.5)
ax.scatter(pillar_years, hazards_bp, c='red', s=50, zorder=5)
ax.set_xlabel('Maturity (years)', fontsize=12)
ax.set_ylabel('Hazard Rate (bp)', fontsize=12)
ax.set_title('Piecewise-Constant Hazard Rates', fontsize=13)
ax.grid(True, alpha=0.3)

# 2. Survival curve
ax = axes[1]
times = np.linspace(0, 10, 200)
survs = [hazard_result.survival_curve.survival(REF + relativedelta(days=int(t*365))) * 100 
         for t in times]
ax.plot(times, survs, 'b-', linewidth=2)
ax.fill_between(times, survs, 100, alpha=0.15, color='red', label='Default probability')
ax.scatter(pillar_years, [hazard_result.survival_curve.survival(d)*100 for d in hazard_result.pillar_dates],
           c='red', s=50, zorder=5, label='Calibration pillars')
ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Survival (%)', fontsize=12)
ax.set_title('Calibrated Survival Curve', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# 3. Calibration fit: market vs model prices
ax = axes[2]
tenors = [_bus_days(REF, b.maturity)/252 for b in corporate_bonds]
market = [b.market_price for b in corporate_bonds]
model = hazard_result.fitted_prices
ax.scatter(tenors, market, c='blue', s=80, label='Market', zorder=5)
ax.scatter(tenors, model, c='red', s=80, marker='x', label='Model', zorder=5, linewidths=2)
for i in range(len(tenors)):
    ax.plot([tenors[i], tenors[i]], [market[i], model[i]], 'gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Maturity (years)', fontsize=12)
ax.set_ylabel('Price (per 100)', fontsize=12)
ax.set_title(f'Calibration Fit (RMSE = {hazard_result.rmse_bp:.1f} bp)', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('brazilian_hazard_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: CDS Par Spread Term Structure

Using the calibrated piecewise survival curve to price CDS at standard tenors.

In [ ]:
surv_curve = hazard_result.survival_curve

print(f"{'CDS Tenor':<12s} {'Par Spread':<14s} {'Survival':<12s}")
print("-" * 38)
cds_tenors = [1, 2, 3, 5, 7, 10]
cds_spreads = []
for T in cds_tenors:
    mat = REF + relativedelta(years=T)
    cds = CDS(REF, mat, 0.01, recovery=0.25)
    par = cds.par_spread(cdi_curve, surv_curve)
    q = surv_curve.survival(mat)
    cds_spreads.append(par * 10000)
    print(f"  {T}Y        {par*10000:>8.0f} bp    {q*100:>8.1f}%")

## Step 5: CLN Pricing — Brazilian Corporate

CLN coupon = CDI + 250bp ≈ 17.25% (typical for BBB corporate).
Recovery = 25% (Brazil corporate recovery is lower than US/EU).

In [ ]:
cln = CreditLinkedNote(
    start=REF, end=REF + relativedelta(years=5),
    coupon_rate=0.1725,  # CDI + 250bp
    notional=5_000_000,
    recovery=0.25,
    frequency=Frequency.SEMI_ANNUAL,
    day_count=DayCountConvention.BUS_252,
)

cln_risky = cln.dirty_price(cdi_curve, surv_curve)
rf_surv = SurvivalCurve(REF, [REF + relativedelta(years=i) for i in range(1, 11)], [1.0]*10)
cln_riskfree = cln.dirty_price(cdi_curve, rf_surv)
credit_charge = cln_riskfree - cln_risky

print(f"CLN 5Y, 17.25% coupon, R$ 5M notional, 25% recovery:")
print(f"  Risk-free PV:   R$ {cln_riskfree:>14,.0f}")
print(f"  Risky PV:       R$ {cln_risky:>14,.0f}")
print(f"  Credit charge:  R$ {credit_charge:>14,.0f}  ({credit_charge/cln_riskfree*100:.1f}%)")

# Sensitivity to recovery
print(f"\nRecovery sensitivity:")
for rec in [0.10, 0.25, 0.40, 0.60]:
    cln_r = CreditLinkedNote(REF, REF+relativedelta(years=5), 0.1725, 5_000_000, rec,
                              Frequency.SEMI_ANNUAL, DayCountConvention.BUS_252)
    pv = cln_r.dirty_price(cdi_curve, surv_curve)
    print(f"  R={rec*100:.0f}%: PV = R$ {pv:>12,.0f}  (credit charge = R$ {cln_riskfree-pv:>10,.0f})")

## Step 6: TRS on NTN-F — Unfunded Bond Exposure

TRS receiver gets total return (coupons + price change), pays CDI + spread.

In [ ]:
bond_price = ntn_f_prices["NTN-F 2029"]
trs = TotalReturnSwap(
    underlying=bond_price, notional=10_000_000,
    start=REF, end=REF + relativedelta(years=1),
    repo_spread=0.0150,  # CDI + 150bp
    haircut=0.05,
    recovery=0.25,
)
trs_result = trs.price(cdi_curve)

print(f"TRS on NTN-F 2029 (1Y term):")
print(f"  Bond price:   {bond_price:.4f}")
print(f"  Notional:     R$ {10_000_000:,.0f}")
print(f"  Funding:      CDI + 150bp")
print(f"  Haircut:      5%")
print(f"  TRS PV:       R$ {trs_result.price:,.0f}")

In [ ]:
# Summary dashboard — 4 panels
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. CDI curve (inverted)
ax = axes[0, 0]
years_cdi = [c["bus_days"]/252 for c in di_strip]
rates_cdi = [c["rate"]*100 for c in di_strip]
ax.plot(years_cdi, rates_cdi, 'b-o', linewidth=2, markersize=5)
ax.axhline(y=14.75, color='red', linestyle='--', alpha=0.5)
ax.set_title('CDI Term Structure (Inverted)', fontsize=12)
ax.set_ylabel('DI Rate (%)'); ax.grid(True, alpha=0.3)

# 2. Piecewise hazard + survival
ax = axes[0, 1]
ax2 = ax.twinx()
for i in range(len(pillar_years)):
    x0 = 0 if i == 0 else pillar_years[i-1]
    ax.plot([x0, pillar_years[i]], [hazards_bp[i]]*2, 'r-', linewidth=2.5)
ax.set_ylabel('Hazard (bp)', color='red')
times_s = np.linspace(0.1, 10, 100)
survs_s = [hazard_result.survival_curve.survival(REF + relativedelta(days=int(t*365)))*100 for t in times_s]
ax2.plot(times_s, survs_s, 'b--', linewidth=1.5, alpha=0.7)
ax2.set_ylabel('Survival (%)', color='blue')
ax.set_title('Piecewise Hazard + Survival', fontsize=12)
ax.grid(True, alpha=0.3)

# 3. CDS par spread curve
ax = axes[1, 0]
ax.bar(cds_tenors, cds_spreads, width=0.6, color='#e74c3c', alpha=0.8)
ax.set_xlabel('Tenor (years)'); ax.set_ylabel('Par Spread (bp)')
ax.set_title('CDS Par Spread Term Structure', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# 4. CLN decomposition
ax = axes[1, 1]
labels = ['Risk-Free', 'Risky', 'Credit\nCharge']
vals = [cln_riskfree/1e6, cln_risky/1e6, credit_charge/1e6]
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(labels, vals, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'R${val:.2f}M', ha='center', fontsize=10, fontweight='bold')
ax.set_title('CLN 5Y Decomposition', fontsize=12)
ax.set_ylabel('R$ (millions)'); ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Brazilian Credit Derivatives — Calibration Dashboard', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('brazilian_credit_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Step | Input | Output | pricebook Module |
|---|---|---|---|
| 1. CDI Curve | DI futures (14.75% → 10.80%) | Inverted discount curve | `build_cdi_curve_from_di()` |
| 2. NTN-F Pricing | Sovereign conventions + CDI | Bond dirty prices | `create_sovereign_bond("NTN_F")` |
| 3. Hazard Bootstrap | 4 corporate bonds at 3/5/7/10Y | **Piecewise** hazard rates | `bootstrap_hazard_from_bonds()` |
| 4. CDS Curve | Calibrated survival curve | Par spreads at 1-10Y | `CDS.par_spread()` |
| 5. CLN | CDI + survival + 17.25% coupon | Risky PV + credit charge | `CreditLinkedNote` |
| 6. TRS | Bond price + CDI + 150bp | TRS PV | `TotalReturnSwap` |

**Brazilian specifics:**
- Recovery = 25% (lower than US/EU 40%)
- Curve inverted (Selic 14.75% → long-end 10.80%)
- Corporate spread: 200-400bp over sovereign
- All BUS/252 compounding